<a href="https://colab.research.google.com/github/deepakk7195/IISC_CDS_DS/blob/Scalable_ML_GenAI/MiniProject_3_PartA_Medical_Q%26A_GPT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Certification Program in Computational Data Science
## A programme by IISc and TalentSprint
### Mini-Project: Medical Q&A using GPT2

## Learning Objectives

At the end of the experiment, you will be able to:

* perform data preprocessing, EDA and feature extraction on the Medical Q&A dataset
* load a pre-trained tokenizer
* finetune a GPT-2 language model for medical question-answering

## Dataset Description

The dataset used in this project is the *Medical Question Answering Dataset* ([MedQuAD](https://github.com/abachaa/MedQuAD/tree/master)). It includes medical question-answer pairs along with additional information, such as the question type, the question *focus*, its UMLS(Unified Medical Language System) details like - Concept Unique Identifier(*CUI*) and Semantic *Type* and *Group*.

To know more about this data's collection, and construction method, refer to this [paper](https://bmcbioinformatics.biomedcentral.com/articles/10.1186/s12859-019-3119-4).

The data is extracted and is in CSV format with below features:

- **Focus**: the question focus
- **CUI**: concept unique identifier
- **SemanticType**
- **SemanticGroup**
- **Question**
- **Answer**

## Part-A: Grading = 10 Points

## Information

Healthcare professionals often have to refer to medical literature and documents while seeking answers to medical queries. Medical databases or search engines are powerful resources of upto date medical knowledge. However, the existing documentation is large and makes it difficult for professionals to retrieve answers quickly in a clinical setting. The problem with search engines and informative retrieval engines is that these systems return a list of documents rather than answers. Instead, healthcare professionals can use question answering systems to retrieve short sentences or paragraphs in response to medical queries. Such systems have the biggest advantage of generating answers and providing hints in a few seconds.

### Problem Statement

Fine-tune gpt2 model on medical-question-answering-dataset for performing response generation for medical queries.

Please refer to ***M6 Assignment-1 Fine-tune GPT2*** to get familiar with how to load pre-trained gpt2 tokenizer and model.

### Import required packages

In [ ]:
!pip -q install -U accelerate
!pip -q install -U transformers
!pip -q install torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 5.8 MB/s eta 0:00:00


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TextDataset, DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments

import warnings
warnings.filterwarnings('ignore')

In [ ]:
#@title Download the dataset
!wget -q https://cdn.iisc.talentsprint.com/AIandMLOps/MiniProjects/Datasets/MedQuAD.csv
!ls | grep ".csv"

MedQuAD.csv


**Exercise 1: Read the MedQuAD.csv dataset**

**Hint:** pd.read_csv()

In [ ]:
df = pd.read_csv("MedQuAD.csv")
df.shape

(16412, 6)

In [ ]:
df.head()

,Focus,CUI,SemanticType,SemanticGroup,Question,Answer
0,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,What is (are) Adult Acute Lymphoblastic Leukem...,Key Points - Adult acute lymphoblastic leukemi...
1,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,What are the symptoms of Adult Acute Lymphobla...,"Signs and symptoms of adult ALL include fever,..."
2,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,How to diagnose Adult Acute Lymphoblastic Leuk...,Tests that examine the blood and bone marrow a...
3,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,What is the outlook for Adult Acute Lymphoblas...,Certain factors affect prognosis (chance of re...
4,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,Who is at risk for Adult Acute Lymphoblastic L...,Previous chemotherapy and exposure to radiatio...


### Pre-processing and EDA

**Exercise 2: Perform below operations on the dataset [0.5 Mark]**

- Handle missing values
- Remove duplicates from data considering `Question` and `Answer` columns

- **Handle missing values**

In [ ]:
# YOUR CODE HERE

In [ ]:
# Drop missing values
# YOUR CODE HERE
def handle_missing_values(data):
  """
  Drops rows with missing values from a pandas DataFrame.

  Args:
      data: A pandas DataFrame.

  Returns:
      A new pandas DataFrame with missing values dropped.
  """
  return data.dropna()

df = handle_missing_values(df)


In [ ]:
print(df.shape)

(15810, 6)


- **Remove duplicates from data considering `Question` and `Answer` columns**

In [ ]:
# Check duplicates
# YOUR CODE HERE
def remove_duplicate_qa(data):
  """
  Removes duplicate rows from a pandas DataFrame considering only the Question and Answer columns.

  Args:
      data: A pandas DataFrame.

  Returns:
      A new pandas DataFrame with duplicates removed based on Question and Answer.
  """
  return data.drop_duplicates(subset=["Question", "Answer"])

In [ ]:
# Remove duplicates based on Question and Answer
df = remove_duplicate_qa(df)


In [ ]:
print(df.shape)

(15762, 6)


In [ ]:
# Drop duplicates
# YOUR CODE HERE

In [ ]:
# Check duplicates
# YOUR CODE HERE

In [ ]:
df.head()

,Focus,CUI,SemanticType,SemanticGroup,Question,Answer
0,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,What is (are) Adult Acute Lymphoblastic Leukem...,Key Points - Adult acute lymphoblastic leukemi...
1,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,What are the symptoms of Adult Acute Lymphobla...,"Signs and symptoms of adult ALL include fever,..."
2,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,How to diagnose Adult Acute Lymphoblastic Leuk...,Tests that examine the blood and bone marrow a...
3,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,What is the outlook for Adult Acute Lymphoblas...,Certain factors affect prognosis (chance of re...
4,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,Who is at risk for Adult Acute Lymphoblastic L...,Previous chemotherapy and exposure to radiatio...


**Exercise 3: Display the category name, and the number of records belonging to top 100 categories of `Focus` column [1 Mark]**

In [ ]:
# YOUR CODE HERE
def analyze_focus_categories(data):
  """
  Analyzes the Focus column and displays the top 100 category names and their record counts.

  Args:
      data: A pandas DataFrame containing a 'Focus' column.

  Returns:
      None (prints the results directly).
  """
  # Count occurrences of each category in the Focus column
  focus_counts = data['Focus'].value_counts()

  # Select the top 100 categories (or all if less than 100)
  top_focus_counts = focus_counts.head(min(100, len(focus_counts)))

  # Print category names and record counts
  print("Category Name\tRecord Count")
  print("---------------------------")
  for category, count in top_focus_counts.items():
    print(f"{category}\t{count}")


In [ ]:
# Top 100 Focus categories names
# YOUR CODE HERE
analyze_focus_categories(df)

Category Name	Record Count
---------------------------
Breast Cancer	53
Prostate Cancer	43
Stroke	35
Skin Cancer	34
Alzheimer's Disease	30
Colorectal Cancer	29
Lung Cancer	29
Heart Failure	28
Heart Attack	28
High Blood Cholesterol	28
High Blood Pressure	27
Parkinson's Disease	25
Leukemia	22
Osteoporosis	21
Shingles	21
Hemochromatosis	20
Age-related Macular Degeneration	20
Diabetes	20
Gum (Periodontal) Disease	19
Diabetic Retinopathy	19
Psoriasis	19
Kidney Disease	17
Dry Mouth	16
COPD	16
Cataract	16
Balance Problems	16
Gout	15
Wilson Disease	15
Medicare and Continuing Care	15
Prescription and Illicit Drug Abuse	15
Glaucoma	15
Rheumatoid Arthritis	14
Neuroblastoma	14
Short Bowel Syndrome	14
Problems with Taste	14
Narcolepsy	14
Endometrial Cancer	14
Osteoarthritis	14
Kidney Dysplasia	13
Problems with Smell	13
Dry Eye	13
Pituitary Tumors	13
Anxiety Disorders	13
Urinary Tract Infections in Children	13
Peripheral Arterial Disease (P.A.D.)	13
Surviving Cancer	13
Amyloidosis and Kidney Disease

### Create Training and Validation set

**Exercise 4: Create training and validation set [2 Marks]**

- Consider 4 samples per `Focus` category, for each top 100 categories, from the dataset (It will give 400 samples for training)

- Consider 1 sample per `Focus` category (different from training set), for each top 100 categories, from the dataset (It will give 100 samples for validation)

In [ ]:
# YOUR CODE HERE
def create_train_validation_split(data, num_train_samples_per_category=4, num_validation_samples_per_category=1):
  """
  Splits the data into training and validation sets based on Focus categories.

  Args:
      data: A pandas DataFrame containing a 'Focus' column.
      num_train_samples_per_category: Number of samples per category for training (default: 4).
      num_validation_samples_per_category: Number of samples per category for validation (default: 1).

  Returns:
      A tuple containing two DataFrames: training set and validation set.
  """
  # Count occurrences of each category in the Focus column
  focus_counts = data['Focus'].value_counts()

  # Select the top 100 categories (or all if less than 100)
  top_focus_counts = focus_counts.head(min(100, len(focus_counts)))

  # Define empty DataFrames for training and validation sets
  training_data = pd.DataFrame(columns=data.columns)
  validation_data = pd.DataFrame(columns=data.columns)

  for category, count in top_focus_counts.items():
    # Shuffle the data for each category
    category_data = data[data['Focus'] == category].sample(frac=1)

    # Select training samples
    training_samples = category_data.iloc[:num_train_samples_per_category]
    training_data = pd.concat([training_data, training_samples], ignore_index=True)

    # Select validation samples (different from training set)
    if count > num_validation_samples_per_category:
      validation_samples = category_data[~category_data.index.isin(training_samples.index)].iloc[:num_validation_samples_per_category]
    else:
      validation_samples = category_data.iloc[:num_validation_samples_per_category]
    validation_data = pd.concat([validation_data, validation_samples], ignore_index=True)

  return training_data, validation_data

In [ ]:
training_data, validation_data = create_train_validation_split(df)

In [ ]:
training_data.shape

(400, 6)

In [ ]:
validation_data.shape

(100, 6)

### Pre-process `Question` and `Answer` text

**Exercise 5: Perform below tasks: [1.5 Marks]**

- Combine `Question` and `Answer` for train and validation data as shown below:
    - sequence = *'\<question\>' + question-text + '\<answer\>' + answer-text*

- Join the combined text using '\n' into a single string for training and validation separately

- Save the training and validation strings as separate text files

- **Combine Question and Answer for train and val data**

In [ ]:
# YOUR CODE HERE

- **Join the combined text using '\n' into a single string for training and validation separately**

In [ ]:
# YOUR CODE HERE
def prepare_text_data(data, training_file="training_data.txt", validation_file="validation_data.txt"):
  """
  Combines Question and Answer columns, creates training and validation strings, and saves them to files.

  Args:
      data: A pandas DataFrame containing 'Question' and 'Answer' columns.
      training_file: Name of the file to save training data (default: training_data.txt).
      validation_file: Name of the file to save validation data (default: validation_data.txt).
  """
  # Define formatting strings for combining text
  question_start = "<question>"
  answer_start = "<answer>"
  delimiter = "\n"

  # Split data into training and validation sets (assuming you have them already)
  training_data = data  # Replace with your actual training data
  validation_data = data  # Replace with your actual validation data

  # Prepare training data string
  training_text = ""
  for index, row in training_data.iterrows():
    question = row["Question"]
    answer = row["Answer"]
    combined_text = question_start + question + answer_start + answer + delimiter
    training_text += combined_text

  # Save training data to a text file
  with open(training_file, "w") as f:
    f.write(training_text)

  # Prepare validation data string
  validation_text = ""
  for index, row in validation_data.iterrows():
    question = row["Question"]
    answer = row["Answer"]
    combined_text = question_start + question + answer_start + answer + delimiter
    validation_text += combined_text

  # Save validation data to a text file
  with open(validation_file, "w") as f:
    f.write(validation_text)

In [ ]:
prepare_text_data(training_data, "training_data.txt", "validation_data.txt")

- **Save the training and validation strings as text files**

In [ ]:
# YOUR CODE HERE

**Exercise 6: Load pre-trained GPT2Tokenizer [0.5 Mark]**

- Use checkpoint = "gpt2"

In [ ]:
# YOUR CODE HERE
# Set up the tokenizer
checkpoint = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(checkpoint)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

**Exercise 7: Tokenize train and validation data and form TextDataset objects [0.5 Mark]**

- Use the loaded pre-trained tokenizer
- Use training and validation data saved in text files

In [ ]:
# YOUR CODE HERE
# Tokenize train text
train_dataset = TextDataset(tokenizer=tokenizer, file_path="training_data.txt", block_size=128)

# Tokenize validation text
val_dataset = TextDataset(tokenizer=tokenizer, file_path="validation_data.txt", block_size=128)

In [ ]:
# Length of train and validation set
len(train_dataset), len(val_dataset)
train_dataset[0].shape, val_dataset[0].shape

(torch.Size([128]), torch.Size([128]))

**Exercise 8: Create a DataCollator object [0.5 Mark]**

In [ ]:
# YOUR CODE HERE
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, return_tensors="pt")

**Exercise 9: Load pre-trained GPT2LMHeadModel [0.5 Mark]**

In [ ]:
# YOUR CODE HERE
model = GPT2LMHeadModel.from_pretrained(checkpoint)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

**Exercise 10: Fine-tune GPT2 Model [1 Mark]**

- Specify training arguments and create a TrainingArguments object (Use 30 epochs)

- Train a GPT-2 model using the provided training arguments

- Save the resulting trained model and tokenizer to a specified output directory

In [ ]:
# Set up the training arguments

# YOUR CODE HERE
model_output_path = "/content/gpt_model"

training_args = TrainingArguments(
    output_dir = model_output_path,
    overwrite_output_dir = True,
    per_device_train_batch_size = 4, # try with 2
    per_device_eval_batch_size = 4,  #  try with 2
    num_train_epochs = 100,
    save_steps = 1_000,
    save_total_limit = 2,
    logging_dir = './logs',
    )

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.21.0`: Please run `pip install transformers[torch]` or `pip install accelerate -U`

In [ ]:
# Train the model
# YOUR CODE HERE

# Save the model
# YOUR CODE HERE

# Save the tokenizer
# YOUR CODE HERE
trainer = Trainer(
    model = model,
    args = training_args,
    data_collator = data_collator,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
)

trainer.train()

# Save the model
trainer.save_model(model_output_path)

# Save the tokenizer
tokenizer.save_pretrained(model_output_path)

Step,Training Loss
500,2.525500
1000,1.951200
1500,1.573900
2000,1.263900
2500,1.012100
3000,0.808200
3500,0.641800
4000,0.514600
4500,0.414500
5000,0.333000


**Exercise 11: Test Model with user input prompts [1 Mark]**

- Create `generate_response()` function that takes a trained *model*, *tokenizer*, and a *prompt* string as input and generates a response using the GPT-2 model

- Test it with some user input prompts

In [ ]:
# YOUR CODE HERE
def generate_response(model, tokenizer, prompt, max_length=100):

    input_ids = tokenizer.encode(prompt, return_tensors="pt")      # 'pt' for returning pytorch tensor

    # Create the attention mask and pad token id
    attention_mask = torch.ones_like(input_ids)
    pad_token_id = tokenizer.eos_token_id

    output = model.generate(
        input_ids,
        max_length=max_length,
        num_return_sequences=1,
        attention_mask=attention_mask,
        pad_token_id=pad_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
# Load the fine-tuned model and tokenizer

# YOUR CODE HERE
checkpoint = "gpt2"
model_output_path = "/content/gpt_model"

model = GPT2LMHeadModel.from_pretrained(checkpoint)

model_tuned = GPT2LMHeadModel.from_pretrained(model_output_path)
model_tuned_tokenizer = GPT2Tokenizer.from_pretrained(model_output_path)

OSError: /content/gpt_model does not appear to have a file named config.json. Checkout 'https://huggingface.co//content/gpt_model/tree/main' for available files.

In [ ]:
# Testing with given prompt 1

# YOUR CODE HERE
prompt = "What is Adult Acute Lymphoblastic?"  # Replace with your desired prompt
response = generate_response(model_tuned, model_tuned_tokenizer, prompt)
print("Generated response:", response)

In [ ]:
# Testing with given prompt 2

# YOUR CODE HERE
prompt = "What is Adult Acute Myeloid Leukemia?"  # Replace with your desired prompt
response = generate_response(model_tuned, model_tuned_tokenizer, prompt)
print("Generated response:", response)

**Exercise 12: Compare the performance of a *GPT2 model* with the *GPT2 model fine-tuned* on MedQuAD data [1 Mark]**

- Load another pre-trained GPT2LMHeadModel and do not fine-tune it

- To generate response using the untuned model, pass it as a parameter to `generate_response()` function

- Test both models (fine-tuned and untuned) with below user input prompts:

    - "What precautions to take for a healthy life?"
    - "What to do after being diagnosed with cancer?"
    - "What to do when feeling sick?"

In [ ]:
prompt1 = "What precautions to take for a healthy life"
prompt2 = "What to do after being diagnosed with cancer?"
prompt3 = "What to do when feeling sick?"

In [ ]:
# Load a pre-trained GPT2 model, do not finetune it with MedQuAD data
# YOUR CODE HERE

model_untuned = GPT2LMHeadModel.from_pretrained(checkpoint)
model_untuned_tokenizer = GPT2LMHeadModel.from_pretrained(checkpoint)

In [ ]:
# Testing with finetuned model: prompt 1
# YOUR CODE HERE

response = generate_response(model_tuned, model_tuned_tokenizer, prompt1)
print("Generated response:", response)

In [ ]:
# Testing with untuned model: prompt 1
# YOUR CODE HERE

response = generate_response(model_untuned, model_untuned_tokenizer, prompt1)
print("Generated response:", response)

In [ ]:
# Testing with finetuned model: prompt 2
# YOUR CODE HERE

response = generate_response(model_tuned, model_tuned_tokenizer, prompt2)
print("Generated response:", response)

In [ ]:
# Testing with untuned model: prompt 2
# YOUR CODE HERE

response = generate_response(model_untuned, model_untuned_tokenizer, prompt2)
print("Generated response:", response)

In [ ]:
# Testing with finetuned model: prompt 3
# YOUR CODE HERE

response = generate_response(model_tuned, model_tuned_tokenizer, prompt3)
print("Generated response:", response)

In [ ]:
# Testing with untuned model: prompt 3
# YOUR CODE HERE

response = generate_response(model_untuned, model_untuned_tokenizer, prompt3)
print("Generated response:", response)